# OOB swaps demo — htmx v4

## Changes from htmx v2 (`oob.py`)

**1. App init:** `fast_app(htmx=False, htmx4=True)` — loads htmx v4 instead of v2.

**2. Selector-based OOB → `Partial`:**  
In v2, you could write `hx_swap_oob='afterend:#first'` to target an arbitrary element with a CSS selector. In v4, it's recommended to use `<hx-partial>` tags, which are more explicit:

| v2 | v4 |
|----|-----|
| `Div(P('thing C'), hx_swap_oob='afterend:#first')` | `Partial(P('thing C'), hx_target='#first', hx_swap='afterend')` |
| `Div(P('thing F'), hx_swap_oob='innerHTML', id='second')` | `Partial(P('thing F'), hx_target='#second', hx_swap='innerHTML')` |

**3. Plain ID OOB unchanged:**  
`P('thing J', hx_swap_oob='true', id='third')` is kept as-is. v4 still recommends `hx_swap_oob=True` for simple same-element ID replacement.

**4. Swap ordering difference:**  
v2 processes OOB swaps before the primary swap. v4 processes: primary → OOB → partials. This means elements inserted with `afterend` may appear in a different order relative to the primary content when migrating.

In [1]:
from fasthtml.common import *
from fasthtml.jupyter import *

In [2]:
app,rt = fast_app(htmx=False, htmx4=True)

@rt
def link():
    return (
        P('thing A'),
        P('thing B'),
        HxPartial(P('thing C'), hx_target='#first', hx_swap='afterend'),
        HxPartial(P('thing D'), hx_target='#first', hx_swap='afterend'),
        HxPartial(P('thing E'), hx_target='#first', hx_swap='afterend'),
        HxPartial(P('thing F'), hx_target='#second', hx_swap='innerHTML'),
        HxPartial(P('thing G'), hx_target='#second', hx_swap='beforeend'),
        HxPartial(P('thing H'), hx_target='#second', hx_swap='beforeend'),
        HxPartial(P('thing I'), hx_target='#second', hx_swap='beforeend'),
        P('thing J', hx_swap_oob='true', id='third'),  # plain ID swap, oob is fine
    )

@rt
def index():
    cts = (
        Button('click', hx_target='#first', hx_get=link, hx_swap='afterend'),
        Grid(
            Div( H3('first'),  Div(id='first' ) ),
            Div( H3('second'), Div(id='second') ),
            Div( H3('third'),  Div(id='third' ) )
        )
    )
    return Titled('HTMX swaps demo', *cts)

srv = JupyUvi(app)